# IEEE Challenge — Flood Prediction Data Pipeline

Collects daily rainfall, soil-moisture, and river-discharge data for two catchments feeding the Niger–Benue system, then merges everything into one flood-prediction dataset.

| # | Catchment | Sources | Purpose |
|---|-----------|---------|---------|
| 1–3 | Kogi State | CHIRPS, IMERG, SMAP | Primary LSTM inputs |
| 4 | Lagdo Dam catchment (Adamawa/Taraba, NG + N. Cameroon) | CHIRPS, IMERG, SMAP | Upstream proxy / `lagdo_risk_flag` |
| 5 | Lokoja | Open-Meteo Flood API | River discharge (target/feature) |
| 6 | — | merge of all above | Final `IEEE flood prediction data.csv` |

> **Note:** Cell outputs (progress bars, widget state) from the original run have been cleared. Re-run top to bottom to regenerate the CSVs. This assumes the per-source CSVs referenced in Part 5 (`kogi_*`, `lagdo_*`) already exist on disk from earlier runs of Parts 1–4.

## Part 1 — CHIRPS Daily Rainfall (Kogi State)

Downloads gzipped daily GeoTIFFs from the CHIRPS Africa archive, clips each to the Kogi State bounding box, and averages rainfall over that box.

In [ ]:
!pip install requests rasterio numpy pandas tqdm --break-system-packages
!pip install requests xarray h5netcdf netCDF4 numpy pandas tqdm --break-system-packages

### Config, helpers, and the per-year download routine

In [ ]:
import gzip
import io
import os
import time
from datetime import date, timedelta

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds
import requests
from tqdm import tqdm

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

BASE_URL = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/africa_daily/tifs/p05"

START_YEAR = 2010
END_YEAR = 2011  # inclusive
Chirps_Dataset = []

# Rough bounding box for Kogi State, Nigeria (lon_min, lat_min, lon_max, lat_max)
# Buffered slightly beyond the state's actual borders to be safe.
# Replace with the GIS person's exact kogi_lgas.geojson bounds once available.
KOGI_BBOX = (5.40, 6.30, 7.80, 8.90)  # (min_lon, min_lat, max_lon, max_lat)

RAW_CACHE_DIR = "chirps_raw_cache"   # temp per-day GeoTIFFs (deleted after use)
OUTPUT_CSV = f"kogi_chirps_rainfall_{START_YEAR}_{END_YEAR}.csv"

REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 5

# ---------------------------------------------------------------------------


def daterange(start_year: int, end_year: int):
    d = date(start_year, 1, 1)
    end = date(end_year, 12, 31)
    while d <= end:
        yield d
        d += timedelta(days=1)


def build_url(d: date) -> str:
    return f"{BASE_URL}/{d.year}/chirps-v2.0.{d.year}.{d.month:02d}.{d.day:02d}.tif.gz"


def download_and_clip(d: date, bbox) -> float | None:
    """
    Downloads one day's gzipped GeoTIFF into memory, decompresses it,
    clips to bbox, and returns the mean rainfall (mm) over that box.
    Returns None if the file is missing or unreadable (some dates are
    occasionally absent from the archive).
    """
    url = build_url(d)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 404:
                return None  # missing date, skip
            resp.raise_for_status()

            # Decompress gzip in memory
            raw_tif = gzip.decompress(resp.content)

            with rasterio.io.MemoryFile(raw_tif) as memfile:
                with memfile.open() as src:
                    window = from_bounds(*bbox, transform=src.transform)
                    data = src.read(1, window=window)

                    nodata = src.nodata
                    if nodata is not None:
                        data = np.where(data == nodata, np.nan, data)

                    if data.size == 0 or np.all(np.isnan(data)):
                        return None

                    return float(np.nanmean(data))

        except Exception as e:  # noqa: BLE001
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS)
            continue

    print(f"  [WARN] Failed {d.isoformat()} after {MAX_RETRIES} attempts: {last_err}")
    return None


def main(START=START_YEAR, END=END_YEAR, Data=Chirps_Dataset):
    os.makedirs(RAW_CACHE_DIR, exist_ok=True)

    dates = list(daterange(START, END))
    print(f"Fetching {len(dates)} days of CHIRPS rainfall for Kogi State "
          f"({START}-{END})...")

    records = []
    for d in tqdm(dates, desc="Downloading + clipping"):
        mean_rainfall_mm = download_and_clip(d, KOGI_BBOX)
        records.append({"date": d.isoformat(), "rainfall_mm": mean_rainfall_mm})

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"])

    missing = df["rainfall_mm"].isna().sum()
    if missing:
        print(f"\n[NOTE] {missing} day(s) could not be retrieved and are NaN. "
              f"Consider interpolating (df['rainfall_mm'].interpolate()) "
              f"before feeding into the LSTM.")

    print(f"\nDone. Saved {len(df)} rows")
    Data.append(df)

### Run one process per year (parallel download)

In [ ]:
from multiprocessing import Manager, Process

if __name__ == "__main__":
    with Manager() as manager:
        # Create a proxy list that lives in a shared manager process
        shared_list = manager.list()

        # Modify main() to accept shared_list and do: shared_list.append(df)
        processes = [
            Process(target=main, args=(yr, yr + 1, shared_list))
            for yr in range(2010, 2023)
        ]

        for p in processes:
            p.start()
        for p in processes:
            p.join()

        # Convert back to a standard Python list
        Chirps_Dataset = list(shared_list)

### Combine per-year results, dedupe, sort, and save

In [ ]:
Master = Chirps_Dataset[0]

for i in Chirps_Dataset[1:]:
    Master = pd.concat([Master, i], ignore_index=True)

Master = Master.drop_duplicates()
Master.sort_values(by='date', inplace=True)
Master.reset_index(drop=True, inplace=True)

Master.to_csv("kogi_chirps_rainfall_2010-2023.csv", index=False)
!cp "/content/kogi_chirps_rainfall_2010-2023.csv" "/content/drive/MyDrive/IEEE challenge 2026/"

## Part 2 — NASA GPM IMERG Daily Rainfall (Kogi / Niger–Benue Basin)

Uses `earthaccess` to authenticate with NASA Earthdata, search and download IMERG Final daily granules over the basin bounding box, then clips and compiles them into a CSV.

In [ ]:
pip install earthaccess xarray h5netcdf numpy pandas tqdm --break-system-packages

In [ ]:
"""
download_imerg_earthaccess.py

Downloads NASA GPM IMERG Final daily rainfall data for the Niger-Benue
basin using the official `earthaccess` library, then clips each granule
locally and compiles a tidy CSV.

Same approach as download_smap_earthaccess.py -- avoids the custom
requests-session + link-list route entirely, so there's no BOM-encoding
or 401/redirect issues to fight. earthaccess handles NASA's auth flow
consistently across data centers (GES DISC here, NSIDC for SMAP).

IMPORTANT: your Earthdata account must have the "NASA GESDISC DATA ARCHIVE"
application authorized (one-time step) or every download will 401:
    1. Go to https://urs.earthdata.nasa.gov/profile
    2. Applications -> Authorized Apps -> Approve More Applications
    3. Search "NASA GESDISC DATA ARCHIVE" and authorize it

Requirements:
    pip install earthaccess xarray h5netcdf netCDF4 numpy pandas tqdm --break-system-packages

On Colab, just run earthaccess.login(strategy="interactive", persist=True)
when prompted -- no environment variables needed.
"""

from pathlib import Path

import earthaccess
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

SHORT_NAME = "GPM_3IMERGDF"
VERSION = "07"

START_DATE = "2016-01-01"
END_DATE = "2023-12-31"

# Niger-Benue basin bounding box (lon_min, lat_min, lon_max, lat_max)
BASIN_BBOX = (4.5, 6.0, 9.5, 10.5)

RAW_DIR = Path("imerg_raw_cache")
RAW_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = "kogi_imerg_rainfall_2.csv"

PRECIP_VAR_CANDIDATES = ["precipitation", "precipitationCal"]

DELETE_RAW_AFTER_PROCESSING = True

# ---------------------------------------------------------------------------


def clip_mean(nc_path: Path, var_candidates: list[str], bbox) -> float | None:
    lon_min, lat_min, lon_max, lat_max = bbox
    try:
        ds = xr.open_dataset(nc_path)
    except Exception as e:  # noqa: BLE001
        print(f"  [WARN] Could not open {nc_path.name}: {e}")
        return None

    var_name = next((v for v in var_candidates if v in ds.variables), None)
    if var_name is None:
        data_vars = list(ds.data_vars)
        var_name = data_vars[0] if data_vars else None
    if var_name is None:
        ds.close()
        return None

    da = ds[var_name]

    lat_name = next((d for d in da.dims if "lat" in d.lower()), None)
    lon_name = next((d for d in da.dims if "lon" in d.lower()), None)
    if lat_name is None or lon_name is None:
        ds.close()
        return None

    try:
        subset = da.sel(
            {lat_name: slice(lat_min, lat_max), lon_name: slice(lon_min, lon_max)}
        )
        if subset.sizes.get(lat_name, 0) == 0:
            subset = da.sel(
                {lat_name: slice(lat_max, lat_min), lon_name: slice(lon_min, lon_max)}
            )
        value = float(subset.mean(skipna=True).values)
    except Exception as e:  # noqa: BLE001
        print(f"  [WARN] Could not clip {nc_path.name}: {e}")
        value = None
    finally:
        ds.close()

    return value


def main():
    print("Logging in to NASA Earthdata via earthaccess...")
    earthaccess.login(strategy="interactive", persist=True)

    print(f"Searching for {SHORT_NAME} v{VERSION} granules "
          f"({START_DATE} to {END_DATE}, basin bbox {BASIN_BBOX})...")

    results = earthaccess.search_data(
        short_name=SHORT_NAME,
        version=VERSION,
        temporal=(START_DATE, END_DATE),
        bounding_box=BASIN_BBOX,
    )
    print(f"Found {len(results)} matching granules.")

    if not results:
        print("No granules found -- check your date range, version, or bbox.")
        return

    print("Downloading granules...")
    downloaded_paths = earthaccess.download(results, str(RAW_DIR))

    records = []
    for path_str in tqdm(downloaded_paths, desc="Clipping"):
        path = Path(path_str)
        mean_val = clip_mean(path, PRECIP_VAR_CANDIDATES, BASIN_BBOX)
        records.append({"source_file": path.name, "rainfall_mm": mean_val})

        if DELETE_RAW_AFTER_PROCESSING:
            try:
                path.unlink()
            except OSError as e:  # noqa: BLE001
                print(f"  [WARN] Could not delete {path.name}: {e}")

    df = pd.DataFrame(records)

    # IMERG filenames look like: 3B-DAY.MS.MRG.3IMERG.20120101-S000000-E235959.V07B.nc4
    df["date"] = df["source_file"].str.extract(r"3IMERG\.(\d{8})-")
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")
    df = df.sort_values("date")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nDone. Saved {len(df)} rows to {OUTPUT_CSV}")
    print(df.head())


if __name__ == "__main__":
    main()

## Part 3 — NASA SMAP Soil Moisture (Kogi, SPL3SMP)

Same `earthaccess` pattern as Part 2, applied to SMAP L3 passive soil-moisture granules.

> **Fixed while formatting:** the original notebook had this section split across cells with a stray, disconnected `2023` cell in between (left over from editing `END_DATE`, which is already set correctly below). That stray cell has been removed and the config/helper/`main()` code merged back into one coherent script, matching the structure of Part 2.

In [ ]:
import os
from pathlib import Path

import earthaccess
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

SHORT_NAME = "SPL3SMP"
VERSION = "009"  # matches "Version 9" confirmed on the NSIDC dataset page

# SMAP only exists from March 2015 onward -- there is no data before this,
# so your 2010-2014 window will simply have no soil moisture values.
START_DATE = "2015-04-01"
END_DATE = "2023-12-31"

# TODO: update basin shapefile bounds once available.
BASIN_BBOX = (4.5, 6.0, 9.5, 10.5)

RAW_DIR = Path("smap_raw_cache")
RAW_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = "kogi_smap_soil_moisture.csv"

# Variable name inside SPL3SMP granules (AM overpass; there's also a PM
# variant "...Retrieval_Data_PM_soil_moisture" if you want both passes)
SOIL_MOISTURE_VAR_CANDIDATES = [
    "Soil_Moisture_Retrieval_Data_AM_soil_moisture",
    "soil_moisture",
]

DELETE_RAW_AFTER_PROCESSING = True

# ---------------------------------------------------------------------------


def clip_mean(nc_path: Path, var_candidates: list[str], bbox) -> float | None:
    """
    SPL3SMP stores its data inside an HDF5 group (not at the root level),
    typically "Soil_Moisture_Retrieval_Data_AM". We try that group first,
    then fall back to the PM group, then to no group at all.
    """
    lon_min, lat_min, lon_max, lat_max = bbox

    groups_to_try = ["Soil_Moisture_Retrieval_Data_AM", "Soil_Moisture_Retrieval_Data_PM", None]

    for group in groups_to_try:
        try:
            ds = xr.open_dataset(nc_path, engine="h5netcdf", group=group, phony_dims="sort")
        except Exception:
            continue

        var_name = next((v for v in ("soil_moisture", *var_candidates) if v in ds.variables), None)
        if var_name is None:
            ds.close()
            continue

        da = ds[var_name]

        try:
            if "latitude" in ds.variables and "longitude" in ds.variables:
                lat = ds["latitude"]
                lon = ds["longitude"]
                mask = (
                    (lat >= lat_min) & (lat <= lat_max) &
                    (lon >= lon_min) & (lon <= lon_max)
                )
                values = da.where(mask)
                fill = da.attrs.get("_FillValue")
                if fill is not None:
                    values = values.where(values != fill)
                value = float(values.mean(skipna=True).values)
            else:
                value = float(da.where(da > -9990).mean(skipna=True).values)
            ds.close()
            return value
        except Exception as e:  # noqa: BLE001
            print(f"  [WARN] Could not clip {nc_path.name} (group={group}): {e}")
            ds.close()
            continue

    print(f"  [WARN] No matching soil moisture variable found in {nc_path.name}")
    return None


def main():
    print("Logging in to NASA Earthdata via earthaccess...")
    earthaccess.login(strategy="interactive", persist=True)

    print(f"Searching for {SHORT_NAME} v{VERSION} granules "
          f"({START_DATE} to {END_DATE}, basin bbox {BASIN_BBOX})...")

    results = earthaccess.search_data(
        short_name=SHORT_NAME,
        version=VERSION,
        temporal=(START_DATE, END_DATE),
        bounding_box=BASIN_BBOX,
    )
    print(f"Found {len(results)} matching granules.")

    if not results:
        print("No granules found -- check your date range, version, or bbox.")
        return

    print("Downloading granules...")
    downloaded_paths = earthaccess.download(results, str(RAW_DIR))

    records = []
    for path_str in tqdm(downloaded_paths, desc="Clipping"):
        path = Path(path_str)
        mean_val = clip_mean(path, SOIL_MOISTURE_VAR_CANDIDATES, BASIN_BBOX)
        records.append({"source_file": path.name, "soil_moisture": mean_val})

        if DELETE_RAW_AFTER_PROCESSING:
            try:
                path.unlink()
            except OSError as e:  # noqa: BLE001
                print(f"  [WARN] Could not delete {path.name}: {e}")

    df = pd.DataFrame(records)

    # SPL3SMP filenames look like: SMAP_L3_SM_P_20180615_R19240_001.h5
    df["date"] = df["source_file"].str.extract(r"_(\d{8})_")
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")
    df = df.sort_values("date")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nDone. Saved {len(df)} rows to {OUTPUT_CSV}")
    print(df.head())


if __name__ == "__main__":
    main()

## Part 4 — Lagdo Dam Catchment (CHIRPS, IMERG, SMAP)

The same three pipelines re-pointed at the **Lagdo Dam catchment** (upper Benue basin: Adamawa/Taraba states, Nigeria, plus adjacent northern Cameroon), to feed the `lagdo_risk_flag` proxy rather than the Kogi LSTM inputs. Column names carry a `_lagdo` suffix so they don't collide with the Kogi columns once merged in Part 5.

### 4a — CHIRPS rainfall, Lagdo catchment

In [ ]:
"""
download_chirps_lagdo.py

Downloads CHIRPS v2.0 Africa daily rainfall (p05, 0.05deg) for the Lagdo
Dam catchment (upper Benue basin: Adamawa/Taraba states, Nigeria, and
adjacent northern Cameroon), clips each day to that bounding box, and
compiles 14 years of daily rainfall into a single tidy CSV.

This is the same proven pipeline as download_chirps_kogi.py -- only the
bounding box and output filename have changed, so it feeds the Lagdo
proxy detection logic (lagdo_risk_flag) rather than the Kogi LSTM inputs.

Source: https://data.chc.ucsb.edu/products/CHIRPS-2.0/africa_daily/tifs/p05/

Requirements:
    pip install requests rasterio numpy pandas tqdm --break-system-packages

Usage:
    python download_chirps_lagdo.py
"""

import gzip
import os
import time
from datetime import date, timedelta

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds
import requests
from tqdm import tqdm

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

BASE_URL = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/africa_daily/tifs/p05"

START_YEAR = 2010
END_YEAR = 2023  # inclusive

# Lagdo Dam catchment: upper Benue basin covering Adamawa & Taraba states
# (Nigeria) and the adjacent Cameroon side that actually drains into the
# reservoir. (min_lon, min_lat, max_lon, max_lat)
LAGDO_BBOX = (11.0, 6.5, 14.5, 10.5)

RAW_CACHE_DIR = "chirps_lagdo_raw_cache"
OUTPUT_CSV = "lagdo_chirps_rainfall_2010_2023.csv"

REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 5

# ---------------------------------------------------------------------------


def daterange(start_year: int, end_year: int):
    d = date(start_year, 1, 1)
    end = date(end_year, 12, 31)
    while d <= end:
        yield d
        d += timedelta(days=1)


def build_url(d: date) -> str:
    return f"{BASE_URL}/{d.year}/chirps-v2.0.{d.year}.{d.month:02d}.{d.day:02d}.tif.gz"


def download_and_clip(d: date, bbox) -> float | None:
    url = build_url(d)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 404:
                return None
            resp.raise_for_status()

            raw_tif = gzip.decompress(resp.content)

            with rasterio.io.MemoryFile(raw_tif) as memfile:
                with memfile.open() as src:
                    window = from_bounds(*bbox, transform=src.transform)
                    data = src.read(1, window=window)

                    nodata = src.nodata
                    if nodata is not None:
                        data = np.where(data == nodata, np.nan, data)

                    if data.size == 0 or np.all(np.isnan(data)):
                        return None

                    return float(np.nanmean(data))

        except Exception as e:  # noqa: BLE001
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS)
            continue

    print(f"  [WARN] Failed {d.isoformat()} after {MAX_RETRIES} attempts: {last_err}")
    return None


def main():
    os.makedirs(RAW_CACHE_DIR, exist_ok=True)

    dates = list(daterange(START_YEAR, END_YEAR))
    print(f"Fetching {len(dates)} days of CHIRPS rainfall for the Lagdo Dam "
          f"catchment ({START_YEAR}-{END_YEAR})...")

    records = []
    for d in tqdm(dates, desc="Downloading + clipping"):
        mean_rainfall_mm = download_and_clip(d, LAGDO_BBOX)
        records.append({"date": d.isoformat(), "rainfall_mm_lagdo": mean_rainfall_mm})

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"])

    missing = df["rainfall_mm_lagdo"].isna().sum()
    if missing:
        print(f"\n[NOTE] {missing} day(s) could not be retrieved and are NaN.")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nDone. Saved {len(df)} rows to {OUTPUT_CSV}")
    print(df.head())


if __name__ == "__main__":
    main()

### 4b — IMERG rainfall, Lagdo catchment

In [ ]:
"""
download_imerg_lagdo.py

Downloads NASA GPM IMERG Final daily rainfall for the Lagdo Dam catchment
(upper Benue basin: Adamawa/Taraba, Nigeria, and adjacent northern
Cameroon) using earthaccess, clips locally, and compiles a tidy CSV.

Same proven pipeline as download_imerg_earthaccess.py -- only the bounding
box and output filename changed, to feed the Lagdo proxy detection logic.

Requirements:
    pip install earthaccess xarray h5netcdf netCDF4 numpy pandas tqdm --break-system-packages

On Colab: earthaccess.login(strategy="interactive", persist=True) will
prompt for your Earthdata username/password directly in the cell.

Reminder: your Earthdata account needs the "NASA GESDISC DATA ARCHIVE"
application authorized (urs.earthdata.nasa.gov/profile -> Applications)
or downloads will fail with 401.
"""

from pathlib import Path

import earthaccess
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

SHORT_NAME = "GPM_3IMERGDF"
VERSION = "07"

START_DATE = "2010-01-01"
END_DATE = "2023-12-31"

# Lagdo Dam catchment (lon_min, lat_min, lon_max, lat_max)
LAGDO_BBOX = (11.0, 6.5, 14.5, 10.5)

RAW_DIR = Path("imerg_lagdo_raw_cache")
RAW_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = "lagdo_imerg_rainfall.csv"

PRECIP_VAR_CANDIDATES = ["precipitation", "precipitationCal"]

DELETE_RAW_AFTER_PROCESSING = True

# ---------------------------------------------------------------------------


def clip_mean(nc_path: Path, var_candidates: list[str], bbox) -> float | None:
    lon_min, lat_min, lon_max, lat_max = bbox
    try:
        ds = xr.open_dataset(nc_path)
    except Exception as e:  # noqa: BLE001
        print(f"  [WARN] Could not open {nc_path.name}: {e}")
        return None

    var_name = next((v for v in var_candidates if v in ds.variables), None)
    if var_name is None:
        data_vars = list(ds.data_vars)
        var_name = data_vars[0] if data_vars else None
    if var_name is None:
        ds.close()
        return None

    da = ds[var_name]

    lat_name = next((d for d in da.dims if "lat" in d.lower()), None)
    lon_name = next((d for d in da.dims if "lon" in d.lower()), None)
    if lat_name is None or lon_name is None:
        ds.close()
        return None

    try:
        subset = da.sel(
            {lat_name: slice(lat_min, lat_max), lon_name: slice(lon_min, lon_max)}
        )
        if subset.sizes.get(lat_name, 0) == 0:
            subset = da.sel(
                {lat_name: slice(lat_max, lat_min), lon_name: slice(lon_min, lon_max)}
            )
        value = float(subset.mean(skipna=True).values)
    except Exception as e:  # noqa: BLE001
        print(f"  [WARN] Could not clip {nc_path.name}: {e}")
        value = None
    finally:
        ds.close()

    return value


def main():
    print("Logging in to NASA Earthdata via earthaccess...")
    earthaccess.login(strategy="interactive", persist=True)

    print(f"Searching for {SHORT_NAME} v{VERSION} granules "
          f"({START_DATE} to {END_DATE}, Lagdo catchment bbox {LAGDO_BBOX})...")

    results = earthaccess.search_data(
        short_name=SHORT_NAME,
        version=VERSION,
        temporal=(START_DATE, END_DATE),
        bounding_box=LAGDO_BBOX,
    )
    print(f"Found {len(results)} matching granules.")

    if not results:
        print("No granules found -- check your date range, version, or bbox.")
        return

    print("Downloading granules...")
    downloaded_paths = earthaccess.download(results, str(RAW_DIR))

    records = []
    for path_str in tqdm(downloaded_paths, desc="Clipping"):
        path = Path(path_str)
        mean_val = clip_mean(path, PRECIP_VAR_CANDIDATES, LAGDO_BBOX)
        records.append({"source_file": path.name, "rainfall_mm_lagdo": mean_val})

        if DELETE_RAW_AFTER_PROCESSING:
            try:
                path.unlink()
            except OSError as e:  # noqa: BLE001
                print(f"  [WARN] Could not delete {path.name}: {e}")

    df = pd.DataFrame(records)
    df["date"] = df["source_file"].str.extract(r"3IMERG\.(\d{8})-")
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")
    df = df.sort_values("date")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nDone. Saved {len(df)} rows to {OUTPUT_CSV}")
    print(df.head())


if __name__ == "__main__":
    main()

### 4c — SMAP soil moisture, Lagdo catchment

In [ ]:
"""
download_smap_lagdo.py

Downloads NASA SMAP SPL3SMP daily soil moisture for the Lagdo Dam
catchment (upper Benue basin: Adamawa/Taraba, Nigeria, and adjacent
northern Cameroon) using earthaccess, clips locally, and compiles a
tidy CSV.

Same proven pipeline as download_smap_earthaccess.py (including the fix
for SMAP's nested HDF5 group structure) -- only the bounding box and
output filename changed.

Requirements:
    pip install earthaccess xarray h5netcdf numpy pandas tqdm --break-system-packages
"""

from pathlib import Path

import earthaccess
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

SHORT_NAME = "SPL3SMP"
VERSION = "009"

# SMAP only exists from March 2015 onward
START_DATE = "2015-04-01"
END_DATE = "2023-12-31"

# Lagdo Dam catchment (lon_min, lat_min, lon_max, lat_max)
LAGDO_BBOX = (11.0, 6.5, 14.5, 10.5)

RAW_DIR = Path("smap_lagdo_raw_cache")
RAW_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = "lagdo_smap_soil_moisture.csv"

SOIL_MOISTURE_VAR_CANDIDATES = [
    "Soil_Moisture_Retrieval_Data_AM_soil_moisture",
    "soil_moisture",
]

DELETE_RAW_AFTER_PROCESSING = True

# ---------------------------------------------------------------------------


def clip_mean(nc_path: Path, var_candidates: list[str], bbox) -> float | None:
    """
    SPL3SMP stores its data inside an HDF5 group (not at the root level),
    typically "Soil_Moisture_Retrieval_Data_AM". Try that group first, then
    the PM group, then no group at all.
    """
    lon_min, lat_min, lon_max, lat_max = bbox

    groups_to_try = ["Soil_Moisture_Retrieval_Data_AM", "Soil_Moisture_Retrieval_Data_PM", None]

    for group in groups_to_try:
        try:
            ds = xr.open_dataset(nc_path, engine="h5netcdf", group=group, phony_dims="sort")
        except Exception:
            continue

        var_name = next((v for v in ("soil_moisture", *var_candidates) if v in ds.variables), None)
        if var_name is None:
            ds.close()
            continue

        da = ds[var_name]

        try:
            if "latitude" in ds.variables and "longitude" in ds.variables:
                lat = ds["latitude"]
                lon = ds["longitude"]
                mask = (
                    (lat >= lat_min) & (lat <= lat_max) &
                    (lon >= lon_min) & (lon <= lon_max)
                )
                values = da.where(mask)
                fill = da.attrs.get("_FillValue")
                if fill is not None:
                    values = values.where(values != fill)
                value = float(values.mean(skipna=True).values)
            else:
                value = float(da.where(da > -9990).mean(skipna=True).values)
            ds.close()
            return value
        except Exception as e:  # noqa: BLE001
            print(f"  [WARN] Could not clip {nc_path.name} (group={group}): {e}")
            ds.close()
            continue

    print(f"  [WARN] No matching soil moisture variable found in {nc_path.name}")
    return None


def main():
    print("Logging in to NASA Earthdata via earthaccess...")
    earthaccess.login(strategy="interactive", persist=True)

    print(f"Searching for {SHORT_NAME} v{VERSION} granules "
          f"({START_DATE} to {END_DATE}, Lagdo catchment bbox {LAGDO_BBOX})...")

    results = earthaccess.search_data(
        short_name=SHORT_NAME,
        version=VERSION,
        temporal=(START_DATE, END_DATE),
        bounding_box=LAGDO_BBOX,
    )
    print(f"Found {len(results)} matching granules.")

    if not results:
        print("No granules found -- check your date range, version, or bbox.")
        return

    print("Downloading granules...")
    downloaded_paths = earthaccess.download(results, str(RAW_DIR))

    records = []
    for path_str in tqdm(downloaded_paths, desc="Clipping"):
        path = Path(path_str)
        mean_val = clip_mean(path, SOIL_MOISTURE_VAR_CANDIDATES, LAGDO_BBOX)
        records.append({"source_file": path.name, "soil_moisture_lagdo": mean_val})

        if DELETE_RAW_AFTER_PROCESSING:
            try:
                path.unlink()
            except OSError as e:  # noqa: BLE001
                print(f"  [WARN] Could not delete {path.name}: {e}")

    df = pd.DataFrame(records)
    df["date"] = df["source_file"].str.extract(r"_(\d{8})_")
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")
    df = df.sort_values("date")

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nDone. Saved {len(df)} rows to {OUTPUT_CSV}")
    print(df.head())


if __name__ == "__main__":
    main()

## Part 5 — Merge Kogi Data + Lokoja River Discharge

Loads every per-source CSV produced above, merges the Kogi rainfall/soil-moisture data into one frame, backfills CHIRPS gaps from IMERG, and joins in daily river-discharge for Lokoja from the Open-Meteo Flood API.

> Two IMERG and two SMAP Kogi files (`..._2.csv` suffix) are loaded because the source data was pulled across two separate runs — both are merged so no days are missed.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/IEEE challenge 2026/kogi_chirps_rainfall_2010-2023.csv")
df2 = pd.read_csv("/content/kogi_smap_soil_moisture.csv")
df3 = pd.read_csv("/content/kogi_imerg_rainfall_2.csv")
df4 = pd.read_csv("/content/kogi_smap_soil_moisture_2.csv")
df5 = pd.read_csv("/content/kogi_imerg_rainfall.csv")
df6 = pd.read_csv("/content/lagdo_imerg_rainfall.csv")
df7 = pd.read_csv("/content/lagdo_imerg_rainfall_2.csv")
df8 = pd.read_csv("/content/lagdo_smap_soil_moisture.csv")
df9 = pd.read_csv("/content/lagdo_smap_soil_moisture_2.csv")
df10 = pd.read_csv("/content/lagdo_chirps_rainfall_2010_2023.csv")

In [ ]:
df3.rename(columns={"rainfall_mm": "rainfall_mm_imerg_kogi"}, inplace=True)
df5.rename(columns={"rainfall_mm": "rainfall_mm_imerg_kogi"}, inplace=True)
df.rename(columns={"rainfall_mm": "rainfall_mm_chirps_kogi"}, inplace=True)

In [ ]:
rainfall_df = df5.merge(df3, how="outer", )
moisture_df = df2.merge(df4, how="outer", ).fillna(0.0000)
df = df.merge(moisture_df.drop(columns=['source_file']), how="left", )
df = df.merge(rainfall_df.drop(columns=['source_file']), how="left", )
df.rename(columns={"soil_moisture": "soil_moisture_kogi"}, inplace=True)
df['rainfall_mm_chirps_kogi'] = df['rainfall_mm_chirps_kogi'].interpolate(limit=3)
df['has_soil_moisture_kogi'] = df['soil_moisture_kogi'].notna()
df['soil_moisture_kogi'] = df['soil_moisture_kogi'].fillna(df['soil_moisture_kogi'].mean())
gap_mask = df['rainfall_mm_chirps_kogi'].isna()
df.loc[gap_mask, 'rainfall_mm_chirps_kogi'] = df.loc[gap_mask, 'rainfall_mm_imerg_kogi']
df['rainfall_source_chirps'] = ~gap_mask  # True = real CHIRPS value, False = IMERG-substituted
df.info()

### Join Lokoja river discharge (Open-Meteo Flood API)

In [ ]:
import requests

# Fetch daily river discharge for Lokoja (Lat: 7.80, Lon: 6.74)
url = "https://flood-api.open-meteo.com/v1/flood?latitude=7.80&longitude=6.74&start_date=2010-01-01&end_date=2023-12-31&daily=river_discharge"
response = requests.get(url).json()

# Convert to DataFrame and join to your existing DataFrame
df_discharge = pd.DataFrame(response["daily"])
df_discharge["date"] = df_discharge["time"]

In [ ]:
df = df.merge(df_discharge, how="left", ).drop(columns='time')

In [ ]:
df

## Part 6 — Merge Lagdo Data, Backfill Gaps, and Save Final Dataset

In [ ]:
df10.rename(columns={"rainfall_mm_lagdo": "rainfall_mm_chirps_lagdo"}, inplace=True)
df6.rename(columns={"rainfall_mm_lagdo": "rainfall_mm_imerg_lagdo"}, inplace=True)
df7.rename(columns={"rainfall_mm_lagdo": "rainfall_mm_imerg_lagdo"}, inplace=True)

In [ ]:
rainfall_df_2 = pd.concat([df6, df7])
moisture_df_2 = pd.concat([df8, df9])

In [ ]:
df = df.merge(rainfall_df_2.drop(columns=['source_file']), how="left", )
df = df.merge(moisture_df_2.drop(columns=['source_file']), how="left", )
df = df.merge(df10, how="left", )
df

In [ ]:
df.info()

### Backfill Lagdo gaps and add source/availability flags (same treatment as Kogi in Part 5)

In [ ]:
# Fix 1: backfill Lagdo CHIRPS gap from Lagdo IMERG, same as you did for Kogi
gap_mask = df['rainfall_mm_chirps_lagdo'].isna()
df.loc[gap_mask, 'rainfall_mm_chirps_lagdo'] = df.loc[gap_mask, 'rainfall_mm_imerg_lagdo']
df['rainfall_source_chirps_lagdo'] = ~gap_mask

# Rename the Kogi flag for clarity now that both regions have one
df = df.rename(columns={'rainfall_source_chirps': 'rainfall_source_chirps_kogi'})

# Fix 2: Lagdo soil moisture -- same flag + neutral fill approach as Kogi,
# but given it's much sparser (RFI, not just pre-2015 gap), the flag matters more here
df['has_soil_moisture_lagdo'] = df['soil_moisture_lagdo'].notna()
df['soil_moisture_lagdo'] = df['soil_moisture_lagdo'].fillna(df['soil_moisture_lagdo'].mean())

In [ ]:
df.info()

### Save final flood-prediction dataset

In [ ]:
df.reset_index(drop=True, inplace=True)
df.to_csv("IEEE flood prediction data.csv", index=False)
!cp "/content/IEEE flood prediction data.csv" "/content/drive/MyDrive/IEEE challenge 2026/"